<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_3_cnn_resnet.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/03_vision/notebooks/sesion_03_3_cnn_resnet.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F03_vision%2Fnotebooks%2Fsesion_03_3_cnn_resnet.ipynb)

# SI7011 · Sesión 03 · 3. CNN modernas: BatchNorm, residuales y augmentation
**Juan David Martínez Vargas · Universidad EAFIT**

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | red con bloques tipo VGG (línea base) | la curva de referencia |
| 2 | `BatchNorm2d` después de cada convolución | entrena más rápido |
| 3 | data augmentation: recortes y espejos | la brecha entre entrenamiento y validación |
| 4 | 20 capas con y sin atajo residual | la pérdida de entrenamiento de la red profunda |
| 5 | global average pooling frente a `Flatten` | parámetros de la cabeza, tamaño de entrada |

Corre en CPU en unos 15 minutos y en GPU en 2–3 (Colab, Kaggle, Lightning o local). Las secciones 1–3 tardan lo mismo; la 4 es la más lenta.

In [ ]:
import time
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
int_epochs = lambda: plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
print('PyTorch', torch.__version__, '· device', device)

## Datos: CIFAR-10, como en los notebooks 1 y 2

In [ ]:
train_set = datasets.CIFAR10('data', train=True, download=True)
test_set = datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes
to_tensor = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float() / 255
X = to_tensor(train_set.data); y = torch.tensor(train_set.targets)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
X_train, y_train = X[perm[:10_000]], y[perm[:10_000]]
X_val, y_val = X[perm[45_000:]], y[perm[45_000:]]
X_test, y_test = to_tensor(test_set.data), torch.tensor(test_set.targets)

mean, std = X_train.mean(dim=(0, 2, 3)), X_train.std(dim=(0, 2, 3))
norm = lambda X: (X - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
X_train, X_val, X_test = norm(X_train), norm(X_val), norm(X_test)
X_train.shape

## Herramientas: el loop, ahora con `augment`
Es el loop del notebook 1 con un solo cambio: `train_epoch` acepta una función `augment` que transforma cada lote **de entrenamiento** antes del paso. La validación y la prueba nunca se aumentan. `fit` registra también la accuracy de entrenamiento, para ver la brecha.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_epoch(model, opt, augment=None, batch_size=128):
    model.train()
    perm = torch.randperm(len(X_train))
    total, correct = 0.0, 0
    for i in range(0, len(X_train), batch_size):
        b = perm[i:i + batch_size]
        xb, yb = X_train[b].to(device), y_train[b].to(device)
        if augment is not None:
            xb = augment(xb)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(b); correct += (logits.argmax(1) == yb).sum().item()
    return total / len(X_train), correct / len(X_train)

@torch.no_grad()
def evaluate(model, X, y, chunk=1000):
    model.eval()
    logits = torch.cat([model(X[i:i + chunk].to(device)).cpu() for i in range(0, len(X), chunk)])
    return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

In [ ]:
def fit(model, epochs=12, lr=1e-3, augment=None):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        loss, acc = train_epoch(model, opt, augment)
        val_loss, val_acc = evaluate(model, X_val, y_val)
        for k, v in zip(hist, (loss, acc, val_loss, val_acc)):
            hist[k].append(v)
    print(f"época {epochs}: loss {loss:.3f} · acc {acc:.3f} · val_acc {val_acc:.3f} · {time.time() - t0:.0f} s")
    return hist

def plot_curves(hists, key='val_acc', title=''):
    for name, h in hists.items():
        plt.plot(range(1, len(h[key]) + 1), h[key], label=name)
    plt.xlabel('época'); plt.ylabel(key); plt.title(title); plt.legend(); int_epochs(); plt.show()

n_params = lambda m: sum(p.numel() for p in m.parameters())

## 1. Línea base: bloques tipo VGG
Un bloque son dos convoluciones de 3×3 con ReLU y un *max pooling* que reduce la resolución a la mitad (diapositiva *Why stacks of 3×3?*). La red apila tres bloques con 16, 32 y 64 canales y termina con *global average pooling* y una capa lineal.

In [ ]:
def conv_relu(c_in, c_out, bn=False):
    layers = [nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=not bn)]
    if bn:
        layers.append(nn.BatchNorm2d(c_out))
    return layers + [nn.ReLU()]

def vgg_block(c_in, c_out, bn=False):
    return nn.Sequential(*conv_relu(c_in, c_out, bn), *conv_relu(c_out, c_out, bn), nn.MaxPool2d(2))

def vgg_net(bn=False, widths=(16, 32, 64)):
    blocks, c = [], 3
    for w in widths:
        blocks.append(vgg_block(c, w, bn)); c = w
    return nn.Sequential(*blocks, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

vgg_net()[0]

In [ ]:
torch.manual_seed(0)
model = vgg_net()
print(f'{n_params(model):,} parámetros')
h_vgg = fit(model)

## 2. BatchNorm en las convoluciones
**Cambia un argumento:** `vgg_net(bn=True)`. Cada convolución va seguida de `nn.BatchNorm2d`, que normaliza cada **canal** con la media y la varianza sobre $N$, $H$ y $W$ (diapositiva *BatchNorm in convolutional layers*). El sesgo de la convolución sobra, porque BatchNorm resta la media.

In [ ]:
torch.manual_seed(0)
model_bn = vgg_net(bn=True)
h_bn = fit(model_bn)
plot_curves({'VGG': h_vgg, 'VGG + BN': h_bn}, key='loss', title='Pérdida de entrenamiento')

In [ ]:
plot_curves({'VGG': h_vgg, 'VGG + BN': h_bn}, title='Accuracy de validación')

Con BatchNorm la pérdida de entrenamiento baja más rápido con el mismo $\eta$: es el mismo efecto de la sesión 02, ahora por canal.

Mira la brecha entre `acc` (entrenamiento) y `val_acc`. Con 10 000 imágenes y 12 épocas la red ya empieza a memorizar:

In [ ]:
plt.plot(h_bn['acc'], label='entrenamiento'); plt.plot(h_bn['val_acc'], label='validación')
plt.xlabel('época'); plt.ylabel('accuracy'); plt.title('VGG + BN'); plt.legend(); plt.show()

## 3. Data augmentation
**Cambia el loop:** `fit(..., augment=augment)`. Cada lote de entrenamiento se transforma al azar con un **espejo horizontal** y un **recorte** de 32×32 tomado de la imagen con 4 píxeles de borde. Le decimos al modelo que la clase no cambia con esas transformaciones (diapositiva *Augmentation encodes invariances*).

In [ ]:
def augment(xb, pad=4):
    flip = torch.rand(len(xb), device=xb.device) < 0.5
    xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)                 # espejo horizontal
    padded = F.pad(xb, (pad, pad, pad, pad), mode='reflect')
    ij = torch.randint(0, 2 * pad + 1, (len(xb), 2))
    return torch.stack([padded[k, :, i:i + 32, j:j + 32] for k, (i, j) in enumerate(ij.tolist())])  # recorte

xb = X_train[:6]
show = lambda t: ((t.cpu() * std.view(3, 1, 1) + mean.view(3, 1, 1)).clamp(0, 1)).permute(1, 2, 0)
fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for k, (a, b) in enumerate(zip(xb, augment(xb))):
    axes[0, k].imshow(show(a)); axes[1, k].imshow(show(b))
for ax in axes.flat: ax.axis('off')
plt.show()

In [ ]:
torch.manual_seed(0)
model_aug = vgg_net(bn=True)
h_aug = fit(model_aug, augment=augment)
plot_curves({'VGG + BN': h_bn, 'VGG + BN + augmentation': h_aug}, title='Accuracy de validación')

In [ ]:
for name, h in {'VGG + BN': h_bn, 'VGG + BN + augmentation': h_aug}.items():
    print(f"{name:26s} acc {h['acc'][-1]:.3f} · val_acc {h['val_acc'][-1]:.3f} · brecha {h['acc'][-1] - h['val_acc'][-1]:+.3f}")

Con augmentation la accuracy de entrenamiento queda **más baja** y la brecha se cierra: el modelo ve una versión distinta de cada imagen en cada época, así que memorizar cuesta más. Es una regularización, como Dropout en la sesión 02, pero construida con un supuesto sobre los datos.

## 4. Profundidad: 20 capas con y sin atajo
Pasamos a bloques residuales como los de ResNet (diapositiva *Residual blocks with convolutions*): dos convoluciones con BatchNorm y un atajo que suma la entrada. Cuando el bloque reduce la resolución, el atajo es una convolución de 1×1 con stride 2.

Con $n = 3$ bloques por etapa y tres etapas son $6n + 2 = 20$ capas, la ResNet-20 para CIFAR de He et al. **Cambia un argumento:** `residual=False` quita los atajos y deja una red **plana** con exactamente las mismas convoluciones.

In [ ]:
class Block(nn.Module):
    def __init__(self, c_in, c_out, stride=1, residual=True):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out),
        )
        self.residual = residual
        self.shortcut = nn.Identity()
        if residual and (stride != 1 or c_in != c_out):
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, a):
        out = self.f(a)
        if self.residual:
            out = out + self.shortcut(a)
        return torch.relu(out)

def resnet(n=3, residual=True, widths=(16, 32, 64)):
    layers = [nn.Conv2d(3, widths[0], 3, padding=1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU()]
    c = widths[0]
    for stage, w in enumerate(widths):
        for j in range(n):
            layers.append(Block(c, w, stride=2 if (stage > 0 and j == 0) else 1, residual=residual)); c = w
    return nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

print(f'ResNet-20: {n_params(resnet()):,} parámetros · plana-20: {n_params(resnet(residual=False)):,}')

In [ ]:
torch.manual_seed(0)
h_plain20 = fit(resnet(residual=False), epochs=8, augment=augment)
torch.manual_seed(0)
model_res = resnet()
h_res20 = fit(model_res, epochs=8, augment=augment)

In [ ]:
plot_curves({'plana-20': h_plain20, 'ResNet-20': h_res20}, key='loss', title='Pérdida de entrenamiento')
plot_curves({'VGG + BN + aug (6 capas)': h_aug, 'plana-20': h_plain20, 'ResNet-20': h_res20}, title='Accuracy de validación')

Las dos redes de 20 capas tienen las mismas convoluciones; la residual solo agrega los atajos (y tres convoluciones de 1×1). Compara primero la **pérdida de entrenamiento**, no la de validación: el problema de la profundidad es de optimización (sesión 02, notebook 4). Con 20 capas la diferencia es moderada; en *Tu turno* puedes subir a $n = 9$ (56 capas) para verla crecer.

## 5. Global average pooling frente a `Flatten`
Nuestras redes terminan con `AdaptiveAvgPool2d(1)`: un número por canal. La alternativa del notebook 1 era aplanar el último mapa de activaciones.

In [ ]:
head_flatten = nn.Linear(64 * 8 * 8, 10)        # VGG con 3 pools: 64 × 4 × 4; aquí, ResNet-20: 64 × 8 × 8
head_gap = nn.Linear(64, 10)
print(f'cabeza con Flatten: {n_params(head_flatten):,} parámetros · con GAP: {n_params(head_gap):,}')
with torch.no_grad():
    model_res.eval()
    print('ResNet-20 con una imagen de 64×64:', tuple(model_res(torch.zeros(1, 3, 64, 64, device=device)).shape))

Con GAP la cabeza es 64 veces más pequeña y la red acepta imágenes de cualquier tamaño: el promedio deja siempre un vector de 64 números. Así funcionan las ResNet preentrenadas del notebook 4.

## Evaluación en prueba
Elegimos por **validación** y evaluamos en prueba una sola vez.

In [ ]:
candidates = {'VGG + BN + aug': (model_aug, h_aug), 'ResNet-20': (model_res, h_res20)}
name = max(candidates, key=lambda k: max(candidates[k][1]['val_acc']))
test_loss, test_acc = evaluate(candidates[name][0], X_test, y_test)
print(f'{name} · prueba: loss {test_loss:.3f} · acc {test_acc:.3f}')

## Tu turno · 10 minutos
1. Sube la profundidad a $n = 9$ (56 capas), con y sin atajos, durante 5 épocas. ¿Crece la diferencia en la pérdida de entrenamiento? (mejor con GPU)
2. Separa los dos efectos de `augment`: corre solo el espejo y solo el recorte. ¿Cuál aporta más?
3. Agrega un espejo **vertical** a `augment`. ¿Ayuda o perjudica en CIFAR-10? ¿Qué supuesto sobre las imágenes estás rompiendo?
4. En `vgg_block`, reemplaza `nn.MaxPool2d(2)` por una convolución con `stride=2`. ¿Cambian los parámetros? ¿Y la accuracy?

**Para pensar.** BatchNorm, augmentation y los atajos residuales aparecieron en la sesión 02 sobre MLP. ¿Cuál de los tres depende de que los datos sean imágenes, y cuál funcionaría igual en Covertype?

In [ ]:
# Experimento 1: n = 9, con y sin atajos

### Referencias
- Simonyan y Zisserman (2015), *Very Deep Convolutional Networks for Large-Scale Image Recognition*, ICLR.
- He et al. (2016), *Deep Residual Learning for Image Recognition*, CVPR (ResNet-20/56 para CIFAR en la sección 4.2).
- Ioffe y Szegedy (2015), *Batch Normalization*, ICML.
- Chollet y Watson (2025), *Deep Learning with Python*, 3.ª ed., caps. 8–9.